# DGP Implementation & Experiment
### Gender Bias in Algorithmic Hiring

This notebook implements the Data Generating Process (DGP) from
Chapter 3 and runs the bias-correction experiment from Chapter 4.

**Structure:**
1. Parameters (Section 2.3 / Table 3.1)
2. Variable legend / codebook
3. Sampling logic for Z and X₁–X₈ (Section 3.2.1–3.2.2)
4. Outcome model, δ(Z, Occupation) and joint calibration of β₀ (Section 3.2.3–3.2.5)
5. Dataset generation — 11-point sweep (Section 3.3)
6. Verification checks (Section 3.4)
7. Saving datasets to CSV
8. Experiment setup (Section 4.1)
9. Feature matrix construction (Section 4.1)
10. Baseline classifier (Section 4.1)
11. Reweighing — pre-processing (Section 4.1)
12. GridSearch — in-processing (Section 4.1)
13. Threshold optimization — post-processing (Section 4.1)
14. Combined comparison table (Section 4.2)
15. Accuracy — supplementary trade-off (Section 4.2)
16. Visualizations (Section 4.2)

All numerical values are traceable to the sources cited in Chapter 2.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import fsolve

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
np.set_printoptions(suppress=True)

## 1. Parameters

In [ ]:
# Z — Protected variable (gender) base rates
# Source: Hangartner et al. (2021) for M/F; Gallup/Jones (2025) for NB
# Raw values sum to 100.5% (source rounding) -> renormalized below
Z_RAW = {"Male": 0.574, "Female": 0.416, "Non-binary": 0.015}
_z_sum = sum(Z_RAW.values())
Z_PROBS = {k: v / _z_sum for k, v in Z_RAW.items()}
Z_LEVELS = ["Male", "Female", "Non-binary"]

# X1 — Education (9 ordered categories)
# Source: CPS ASEC (2024) for M/F; Williams Institute (2024) for NB (48% HS-or-less only)
EDUCATION_LEVELS = [
    "None-8th", "9th-11th", "HS_grad", "SomeCollege", "Associates",
    "Bachelors", "Masters", "Professional", "Doctoral",
]
EDUCATION_MALE = np.array([3.6, 5.5, 29.6, 13.9, 10.2, 23.0, 10.1, 1.5, 2.5]) / 100
EDUCATION_FEMALE = np.array([3.4, 4.6, 26.2, 14.1, 11.6, 24.4, 12.5, 1.3, 2.0]) / 100
# NB: only "48% HS-or-less" (sum of first 3 cats) is directly sourced (Williams Institute 2024).
# Full 9-category NB distribution interpolated at sampling-module stage.
EDUCATION_NB_HS_OR_LESS_TARGET = 0.48

BETA_EDUCATION = 0.37  # standardized ordinal index coefficient
# Standardization moments (population-level, M/F-pooled), used to z-score the ordinal index
EDUCATION_ORDINAL_MEAN = 4.594
EDUCATION_ORDINAL_SD = 1.817

# X2 — Experience (continuous, truncated Normal at 0)
# Source: Blau & Kahn (2017) means; GAO-04-35 (2003) SDs;
#         NB mean via Mincer-formula (Carpenter et al. 2026), NB SD via CV-transfer from GAO
EXPERIENCE_MEAN = {"Male": 17.8, "Female": 16.4, "Non-binary": 14.16}
EXPERIENCE_SD = {"Male": 9.8, "Female": 8.0, "Non-binary": 8.8}

# X->Y: Hangartner et al. (2021) report only a threshold effect, expressed as a relative
# percent effect on the mean contact rate.
EXPERIENCE_THRESHOLD_YEARS = 4
EXPERIENCE_THRESHOLD_RELATIVE_EFFECT = 0.115  # +11.5% relative to reference rate
# Experience enters f(X) only via this threshold dummy.

# X3 — Skills: Technical & Communication (independent Normals, truncated [0,100])
# Source: Rajesh et al. (2025) for M/F means; Basu (2026) ref. dataset for NB means, all SDs,
#         and the X->Y regression. Independence empirically confirmed (r=0.019).
TECHNICAL_MEAN = {"Male": 69.7, "Female": 69.8, "Non-binary": 69.1}
TECHNICAL_SD = {"Male": 13.96, "Female": 13.73, "Non-binary": 13.06}
COMMUNICATION_MEAN = {"Male": 74.8, "Female": 75.0, "Non-binary": 76.6}
COMMUNICATION_SD = {"Male": 10.37, "Female": 10.49, "Non-binary": 10.39}

BETA_TECHNICAL = 1.151
BETA_COMMUNICATION = 0.549

# X4 — Occupation (21 categories)
# Source: Rus et al. (2022) for M/F; Bai, Chum & Liu (2026) 10-cat crosswalk for NB
# Occupation has no direct X->Y term; enters only via moderation (Section 3.2.5)
OCCUPATIONS = [
    "Administration/Secretarial", "Automation/Internet", "Policy/Executive",
    "Security/Defence/Police", "Commercial/Sales", "Consultancy/Advice",
    "Design/Creative/Journalism", "Management", "Financial/Accounting",
    "Financial services", "Catering/Retail", "HR/Training",
    "Procurement/Logistics/Transport", "Legal", "Customer service/Call centre",
    "Marketing/PR/Communications", "Medical/Healthcare", "Education/Research/Science",
    "Other", "Production/Operational", "Technology",
]

# raw % Male, % Female (will be renormalized to sum to 100% each, in sampling module)
OCCUPATION_MALE_RAW = np.array([
    4.31, 4.71, 3.81, 2.20, 8.82, 6.64, 1.83, 6.41, 3.25, 3.26,
    4.24, 2.55, 9.45, 0.82, 1.90, 4.45, 2.28, 3.65, 8.24, 7.39, 9.77,
])
OCCUPATION_FEMALE_RAW = np.array([
    16.75, 0.85, 3.35, 0.88, 6.64, 4.22, 2.48, 3.21, 2.55, 2.99,
    6.06, 5.36, 2.97, 1.85, 7.11, 5.86, 8.54, 6.63, 8.27, 2.54, 0.91,
])
# Female share per occupation (used directly for moderation)
OCCUPATION_FEMALE_SHARE = np.array([
    0.787, 0.147, 0.456, 0.276, 0.417, 0.377, 0.563, 0.323, 0.427, 0.465,
    0.576, 0.667, 0.230, 0.682, 0.780, 0.556, 0.781, 0.633, 0.488, 0.247, 0.081,
])

# Bai, Chum & Liu (2026) crosswalk raw values -> renormalized in sampling module
# Mapping: Rus category -> Bai group -> raw value
OCCUPATION_NB_CROSSWALK_RAW = {
    "Policy/Executive": 0.20, "Management": 0.20,
    "Administration/Secretarial": 6.04, "Consultancy/Advice": 6.04,
    "Financial/Accounting": 6.04, "Financial services": 6.04,
    "HR/Training": 6.04, "Marketing/PR/Communications": 6.04,
    "Automation/Internet": 4.37, "Technology": 4.37,
    "Medical/Healthcare": 1.84,
    "Legal": 6.40, "Education/Research/Science": 6.40,
    "Design/Creative/Journalism": 4.05,
    "Commercial/Sales": 9.13, "Catering/Retail": 9.13, "Customer service/Call centre": 9.13,
    "Security/Defence/Police": 2.11, "Procurement/Logistics/Transport": 2.11,
    "Production/Operational": 0.81,
    "Other": 3.55,
}

# X5 — Age (7 ordered brackets)
# Source: DOL Women's Bureau (2024) for M/F; Williams Institute (2024) for NB (87% <35 only)
AGE_BRACKETS = ["16-19", "20-24", "25-34", "35-44", "45-54", "55-64", "65+"]
AGE_MALE = np.array([3.7, 8.8, 22.2, 22.4, 19.6, 16.2, 7.1]) / 100
AGE_FEMALE = np.array([4.1, 9.4, 22.0, 22.1, 19.6, 16.1, 6.7]) / 100
AGE_NB_UNDER35_TARGET = 0.87  # sum of first 3 brackets

BETA_AGE = -0.35
AGE_ORDINAL_MEAN = 4.21
AGE_ORDINAL_SD = 1.54

# X6 — Motivation (binary)
# Source: Pisanelli (2022) for M/F; pooled for NB
# X->Y: Carlsson et al. (2014)
MOTIVATION_RATE = {"Male": 0.239, "Female": 0.116, "Non-binary": 0.187}
BETA_MOTIVATION = 0.00

# X7 — Leadership (binary)
# Source: BLS Table 11 (2025) for M/F; Fletcher & Swierczynski (2023/2025) for NB
LEADERSHIP_RATE = {"Male": 0.1405, "Female": 0.1133, "Non-binary": 0.06}
BETA_LEADERSHIP = 1.151 * (17.6 / 18.3)  # ~1.11, RF-importance-scaled from Technical

# X8 — Employment Gap (binary)
# Source: Bertrand, Goldin & Katz (2009/2010) for M/F; pooled for NB
EMPLOYMENT_GAP_RATE = {"Male": 0.095, "Female": 0.319, "Non-binary": 0.189}

# beta derived from Njoto et al. (2025) original tables (n=422/group), averaged across
# Callback and Positive-Email stages
_logit = lambda p: np.log(p / (1 - p))
BETA_EMPLOYMENT_GAP = (
    (_logit(31 / 422) - _logit(52 / 422)) + (_logit(39 / 422) - _logit(110 / 422))
) / 2

# Y — Outcome calibration target
# Source: Njoto et al. (2025), Callback rate, "no employment gap" baseline group
Y_BASELINE_RATE_NOMINAL = 52 / 422  # 12.3% nominal target; beta0 is calibrated

# delta(Z, Occupation) — direct discrimination parameter
# Reparametrized as relative odds reduction. See Section 3.2.4/3.2.5.
G_FEMALE_MAX = 0.69  # Pisanelli (2022) own reported relative figure ("69% lower chances")
G_NONBINARY_FIXED = 0.18  # Eames (2024) own reported relative figure

GAMMA_MODERATION = 0.30  # Hangartner et al. (2021), relative percent-of-mean slope
GAMMA_NB = 0.0  # evidenced null (Eames 2024)

# 11-point sweep: 0%-100% of Pisanelli's own upper anchor, in 10% steps
N_SWEEP_STEPS = 11
G_FEMALE_SWEEP = np.linspace(0, 1, N_SWEEP_STEPS) * G_FEMALE_MAX

# Simulation / reproducibility settings
N_PER_DATASET = 300_000

DATASET_SEEDS = {i: 1000 + i for i in range(N_SWEEP_STEPS)}
BASE_SEED = 42

## 2. Variable Legend / Codebook

A reference for what every column in the generated datasets means, its coding,
and its unit.


In [ ]:
# Ordinal category label lookups
EDUCATION_LABELS = {i + 1: lvl for i, lvl in enumerate(EDUCATION_LEVELS)}
AGE_LABELS = {i + 1: br for i, br in enumerate(AGE_BRACKETS)}

# Column-level documentation: dtype, meaning, unit/coding, range
CODEBOOK = {
    "Z": {
        "meaning": "Protected variable: applicant gender",
        "type": "categorical (string)",
        "values": "'Male', 'Female', 'Non-binary'",
    },
    "Education_Level": {
        "meaning": "Highest educational attainment (ordinal)",
        "type": "integer, 1-9",
        "values": "1=None-8th grade, 2=9th-11th grade, 3=HS graduate, 4=Some college (no degree), "
                  "5=Associate's, 6=Bachelor's, 7=Master's, 8=Professional degree, 9=Doctoral",
    },
    "Experience_Years": {
        "meaning": "Years of full-time work experience",
        "type": "continuous (float)",
        "values": "years, truncated at 0 (no upper bound; realistic range ~0-45)",
    },
    "Technical_Score": {
        "meaning": "CV/assessment technical competence score",
        "type": "continuous (float)",
        "values": "0-100 scale (truncated)",
    },
    "Communication_Score": {
        "meaning": "CV/assessment communication competence score",
        "type": "continuous (float)",
        "values": "0-100 scale (truncated)",
    },
    "Occupation": {
        "meaning": "Applied-to occupational category",
        "type": "categorical (string)",
        "values": "one of 21 categories, e.g. 'Technology', 'Administration/Secretarial' "
                  "(see OCCUPATIONS for full list)",
    },
    "Occupation_Female_Share": {
        "meaning": "Share of women among jobseekers in this occupation category "
                   "(Rus et al. 2022); used only for the delta-moderation mechanism, "
                   "not as a predictive feature",
        "type": "continuous (float)",
        "values": "0-1 (proportion)",
    },
    "Age_Bracket": {
        "meaning": "Applicant age bracket (ordinal)",
        "type": "integer, 1-7",
        "values": "1=16-19, 2=20-24, 3=25-34, 4=35-44, 5=45-54, 6=55-64, 7=65+",
    },
    "Motivation": {
        "meaning": "Presence of agentic self-presentation language in the CV "
                   "(e.g. 'ambitious', 'determined' vs. 'supportive', 'cooperative')",
        "type": "binary",
        "values": "0=absent, 1=present",
    },
    "Leadership": {
        "meaning": "Middle- or senior-level managerial responsibility",
        "type": "binary",
        "values": "0=no, 1=yes",
    },
    "Employment_Gap": {
        "meaning": "Presence of a career interruption / employment gap",
        "type": "binary",
        "values": "0=no gap, 1=gap present",
    },
    "Y": {
        "meaning": "Outcome: positive screening/hiring signal ('callback')",
        "type": "binary",
        "values": "0=not selected, 1=selected",
    },
    "P_true": {
        "meaning": "True generating probability P(Y=1) used to draw this row's Y "
                   "(diagnostic column, not an input feature)",
        "type": "continuous (float)",
        "values": "0-1 (probability)",
    },
    "delta_applied": {
        "meaning": "The direct-discrimination log-odds term delta(Z, Occupation) "
                   "actually applied to this row (diagnostic column, not an input feature)",
        "type": "continuous (float)",
        "values": "<= 0 (0 for reference group Male; negative = penalty)",
    },
}


def print_codebook():
    """Pretty-print the full codebook for notebook display."""
    print(f"{'Column':<26} {'Type':<20} Meaning / Coding")
    print("-" * 110)
    for col, info in CODEBOOK.items():
        print(f"{col:<26} {info['type']:<20} {info['meaning']}")
        print(f"{'':<26} {'':<20} -> {info['values']}")
        print()

In [ ]:
print_codebook()

## 3. Sampling: Z and X₁–X₈

Draws the protected variable Z and all eight applicant characteristics
conditional on Z. Includes:
- **Truncated-normal correction**: naively using the target mean/SD as a
  truncated normal's own parameters biases the result after truncation, so the
  underlying (pre-truncation) parameters are numerically solved for.
- **Non-binary interpolation** for Education and Age: the literature provides
  only one aggregate figure for these two variables (e.g. "48% HS-or-less"),
  not a full category-by-category distribution. The M/F-pooled distribution's
  *shape* is used as a reference, rescaled to match the sourced aggregate.
- **Occupation renormalization**: raw source percentages don't sum to exactly
  100% (multi-category membership in the original data), renormalized here.


In [ ]:
# Truncated-normal correction helper
# Given a TARGET mean/SD for the truncated distribution, solve for the
# underlying (pre-truncation) mu*, sigma* such that truncnorm(mu*, sigma*,
# low, high) has exactly that mean/SD. Needed because naively using the
# target mean/SD as the truncnorm's own parameters biases the result
# (truncation shifts the mean/SD away from the input parameters).
def solve_truncnorm_params(target_mean, target_sd, low, high=np.inf):
    def equations(params):
        mu_star, sigma_star = params
        if sigma_star <= 0:
            return [1e6, 1e6]
        a = (low - mu_star) / sigma_star
        b = (high - mu_star) / sigma_star
        dist = stats.truncnorm(a, b, loc=mu_star, scale=sigma_star)
        return [dist.mean() - target_mean, dist.std() - target_sd]

    # initial guess: naive (uncorrected) parameters
    mu0, sigma0 = target_mean, target_sd
    mu_star, sigma_star = fsolve(equations, [mu0, sigma0], full_output=False)
    return mu_star, sigma_star


def sample_truncnorm(n, target_mean, target_sd, low, high, rng):
    mu_star, sigma_star = solve_truncnorm_params(target_mean, target_sd, low, high)
    a = (low - mu_star) / sigma_star
    b = (high - mu_star) / sigma_star
    return stats.truncnorm.rvs(a, b, loc=mu_star, scale=sigma_star, size=n, random_state=rng)


# NB interpolation for Education and Age
# Method: take the M/F-pooled distribution (pooled using this thesis's own
# Z weights, M/F renormalized ignoring NB) as the reference shape, then
# rescale the "low" block to match the one directly-sourced NB aggregate
# figure and the "high" block to the complement, preserving relative shape
# within each block.
def _interpolate_nb_distribution(male_dist, female_dist, low_block_idx, low_target):
    w_m = Z_RAW["Male"] / (Z_RAW["Male"] + Z_RAW["Female"])
    w_f = Z_RAW["Female"] / (Z_RAW["Male"] + Z_RAW["Female"])
    pooled = w_m * male_dist + w_f * female_dist

    n = len(pooled)
    low_idx = np.array(low_block_idx)
    high_idx = np.array([i for i in range(n) if i not in low_block_idx])

    result = np.zeros(n)
    low_sum = pooled[low_idx].sum()
    high_sum = pooled[high_idx].sum()
    result[low_idx] = pooled[low_idx] * (low_target / low_sum)
    result[high_idx] = pooled[high_idx] * ((1 - low_target) / high_sum)
    return result


EDUCATION_NB = _interpolate_nb_distribution(
    EDUCATION_MALE, EDUCATION_FEMALE,
    low_block_idx=[0, 1, 2],  # None-8th, 9th-11th, HS_grad
    low_target=EDUCATION_NB_HS_OR_LESS_TARGET,
)

AGE_NB = _interpolate_nb_distribution(
    AGE_MALE, AGE_FEMALE,
    low_block_idx=[0, 1, 2],  # 16-19, 20-24, 25-34
    low_target=AGE_NB_UNDER35_TARGET,
)

EDUCATION_DIST = {"Male": EDUCATION_MALE, "Female": EDUCATION_FEMALE, "Non-binary": EDUCATION_NB}
AGE_DIST = {"Male": AGE_MALE, "Female": AGE_FEMALE, "Non-binary": AGE_NB}

# Occupation distributions (renormalized to sum to exactly 1.0)
OCCUPATION_MALE = OCCUPATION_MALE_RAW / OCCUPATION_MALE_RAW.sum()
OCCUPATION_FEMALE = OCCUPATION_FEMALE_RAW / OCCUPATION_FEMALE_RAW.sum()

_nb_raw = np.array([OCCUPATION_NB_CROSSWALK_RAW[occ] for occ in OCCUPATIONS])
OCCUPATION_NB = _nb_raw / _nb_raw.sum()

OCCUPATION_DIST = {"Male": OCCUPATION_MALE, "Female": OCCUPATION_FEMALE, "Non-binary": OCCUPATION_NB}


def _norm_p(p):
    """Robustly renormalize a probability vector to sum to exactly 1.0
    (guards against floating-point sums like 0.9999999999998)."""
    p = np.asarray(p, dtype=float)
    return p / p.sum()


# Main sampling function: draws Z, then all X1-X8 conditional on Z
def sample_population(n, seed):
    rng = np.random.default_rng(seed)

    # Z
    z_levels = Z_LEVELS
    z_probs = _norm_p([Z_PROBS[lvl] for lvl in z_levels])
    Z = rng.choice(z_levels, size=n, p=z_probs)

    # containers
    education_level = np.zeros(n, dtype=int)
    experience_years = np.zeros(n)
    technical_score = np.zeros(n)
    communication_score = np.zeros(n)
    occupation = np.empty(n, dtype=object)
    age_bracket = np.zeros(n, dtype=int)
    motivation = np.zeros(n, dtype=int)
    leadership = np.zeros(n, dtype=int)
    employment_gap = np.zeros(n, dtype=int)

    for z_level in z_levels:
        mask = Z == z_level
        n_z = mask.sum()
        if n_z == 0:
            continue

        # X1 Education (categorical, 9 levels; ordinal index 1-9)
        education_level[mask] = rng.choice(
            np.arange(1, 10), size=n_z, p=_norm_p(EDUCATION_DIST[z_level])
        )

        # X2 Experience (truncated normal at 0)
        experience_years[mask] = sample_truncnorm(
            n_z, EXPERIENCE_MEAN[z_level], EXPERIENCE_SD[z_level], low=0, high=np.inf, rng=rng
        )

        # X3a/b Technical & Communication (truncated normal [0,100], independent)
        technical_score[mask] = sample_truncnorm(
            n_z, TECHNICAL_MEAN[z_level], TECHNICAL_SD[z_level], low=0, high=100, rng=rng
        )
        communication_score[mask] = sample_truncnorm(
            n_z, COMMUNICATION_MEAN[z_level], COMMUNICATION_SD[z_level], low=0, high=100, rng=rng
        )

        # X4 Occupation (categorical, 21 levels)
        occupation[mask] = rng.choice(OCCUPATIONS, size=n_z, p=_norm_p(OCCUPATION_DIST[z_level]))

        # X5 Age (categorical, 7 levels; ordinal index 1-7)
        age_bracket[mask] = rng.choice(np.arange(1, 8), size=n_z, p=_norm_p(AGE_DIST[z_level]))

        # X6 Motivation (binary)
        motivation[mask] = rng.binomial(1, MOTIVATION_RATE[z_level], size=n_z)

        # X7 Leadership (binary)
        leadership[mask] = rng.binomial(1, LEADERSHIP_RATE[z_level], size=n_z)

        # X8 Employment Gap (binary)
        employment_gap[mask] = rng.binomial(1, EMPLOYMENT_GAP_RATE[z_level], size=n_z)

    # occupation female share (needed later for moderation)
    occ_to_female_share = dict(zip(OCCUPATIONS, OCCUPATION_FEMALE_SHARE))
    occupation_female_share = np.array([occ_to_female_share[o] for o in occupation])

    import pandas as pd
    df = pd.DataFrame({
        "Z": Z,
        "Education_Level": education_level,
        "Experience_Years": experience_years,
        "Technical_Score": technical_score,
        "Communication_Score": communication_score,
        "Occupation": occupation,
        "Occupation_Female_Share": occupation_female_share,
        "Age_Bracket": age_bracket,
        "Motivation": motivation,
        "Leadership": leadership,
        "Employment_Gap": employment_gap,
    })
    return df

**Preview: first 5 rows of a small sample population**

In [ ]:
df_preview = sample_population(n=1000, seed=42)
df_preview.head()

**Sanity checks on the sampling logic:**

In [ ]:
print("Education NB interpolation — sum:", EDUCATION_NB.sum(),
      " HS-or-less share:", round(EDUCATION_NB[:3].sum(), 4), "(target: 0.48)")
print("Age NB interpolation — sum:", AGE_NB.sum(),
      " <35 share:", round(AGE_NB[:3].sum(), 4), "(target: 0.87)")
print("Occupation renorm — Male sum:", OCCUPATION_MALE.sum(),
      " Female sum:", OCCUPATION_FEMALE.sum(), " NB sum:", OCCUPATION_NB.sum())

## 4. Outcome Model, δ(Z, Occupation), and Joint Calibration

### The direct-discrimination term δ

Reparametrized as a **relative odds reduction**:

$$\delta_F(g, o) = \ln(1 - g_F(g,o)), \quad g_F(g,o) = g + \gamma \cdot \max(0, 0.5 - o)$$
$$\delta_M(o) = \ln(1 - g_M(o)), \quad g_M(o) = \gamma \cdot \max(0, o - 0.5)$$
$$\delta_{NB} = \ln(1 - 0.18) \quad \text{(fixed, not swept)}$$

Occupation moderation (γ=0.30, Hangartner et al. 2021) stays structurally active
independent of g since it represents a separate mechanism, not a component of the free
sweep parameter itself.

### The linear predictor and joint calibration

$$\eta = \beta_0 + \sum_k \beta_k \cdot X_k + \beta_{ExpThreshold} \cdot \mathbb{1}[Experience \geq 4] + \delta(Z, Occupation)$$

β₀ and β_ExpThreshold cannot be calibrated independently. Because the model
contains several non-trivial coefficients, the population-average outcome
probability does not equal σ(β₀) at a single reference point (Jensen's
inequality). Both are solved **jointly**: for each candidate β₀, the
threshold coefficient is solved in closed form so that crossing the threshold
reproduces exactly the literature-reported +11.5% relative effect (Hangartner
et al. 2021) *at that β₀*; β₀ itself is then adjusted via bisection search over
a large simulated reference population until the population-average outcome
rate matches the calibration target exactly (12.3%, Njoto et al. 2025).

**Methodological note:** Technical and Communication Score have no
population-level standardization moments reported in the source literature.
These are computed empirically from a large simulated reference population.


In [ ]:
# delta(Z, Occupation) — direct discrimination term
# Relative-odds-reduction reparametrization (Section 3.2.4/3.2.5).
# g_female: the free sweep parameter (0 for the "fair" reference dataset).
# NB uses the fixed g=0.18 in EVERY dataset, including "fair".
# Occupation moderation stays structurally active regardless of g.
def compute_delta(Z, occupation_female_share, g_female):
    n = len(Z)
    delta = np.zeros(n)

    is_male = Z == "Male"
    is_female = Z == "Female"
    is_nb = Z == "Non-binary"

    # Male: reference group base g=0, but still subject to moderation if in a
    # female-dominated occupation
    g_male = GAMMA_MODERATION * np.maximum(0, occupation_female_share - 0.5)
    g_male = np.clip(g_male, 0, 0.999)
    delta[is_male] = np.log(1 - g_male[is_male])

    # Female: free parameter g_female + moderation if in a male-dominated occupation
    g_fem = g_female + GAMMA_MODERATION * np.maximum(0, 0.5 - occupation_female_share)
    g_fem = np.clip(g_fem, 0, 0.999)
    delta[is_female] = np.log(1 - g_fem[is_female])

    # Non-binary: fixed g=0.18, not swept, no occupation moderation (gamma_NB=0)
    delta[is_nb] = np.log(1 - G_NONBINARY_FIXED)

    return delta


# Standardization moments
# Education & Age: analytically known, M/F-pooled (per thesis Section 2.3/3.2.2)
# Technical & Communication: no population-level moments reported in the
# literature for standardization purposes -> computed empirically from a
# large simulated reference population.
_STANDARDIZATION_CACHE = {}


def _compute_technical_communication_moments(n_ref=200_000, seed=999):
    """Empirically estimate population-level mean/SD for Technical and
    Communication scores, using this thesis's own Z mixture weights.
    Cached after first call."""
    if "tech_comm" in _STANDARDIZATION_CACHE:
        return _STANDARDIZATION_CACHE["tech_comm"]
    df_ref = sample_population(n=n_ref, seed=seed)
    moments = {
        "technical_mean": df_ref["Technical_Score"].mean(),
        "technical_sd": df_ref["Technical_Score"].std(),
        "communication_mean": df_ref["Communication_Score"].mean(),
        "communication_sd": df_ref["Communication_Score"].std(),
    }
    _STANDARDIZATION_CACHE["tech_comm"] = moments
    return moments


# Linear predictor f(X) — everything except delta and the Experience-threshold term
def compute_base_linear_predictor(df, tc_moments):
    edu_z = (df["Education_Level"].values - EDUCATION_ORDINAL_MEAN) / EDUCATION_ORDINAL_SD
    age_z = (df["Age_Bracket"].values - AGE_ORDINAL_MEAN) / AGE_ORDINAL_SD
    tech_z = (df["Technical_Score"].values - tc_moments["technical_mean"]) / tc_moments["technical_sd"]
    comm_z = (df["Communication_Score"].values - tc_moments["communication_mean"]) / tc_moments["communication_sd"]

    eta = (
        BETA_EDUCATION * edu_z
        + BETA_TECHNICAL * tech_z
        + BETA_COMMUNICATION * comm_z
        + BETA_AGE * age_z
        + BETA_MOTIVATION * df["Motivation"].values
        + BETA_LEADERSHIP * df["Leadership"].values
        + BETA_EMPLOYMENT_GAP * df["Employment_Gap"].values
    )
    return eta


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


# Joint calibration of beta0 and beta_exp_threshold 
def beta_exp_threshold_given_beta0(beta0):
    """Closed-form: solve for the threshold coefficient such that, AT THE
    REFERENCE POINT (all other terms = 0), crossing the >=4yr threshold
    multiplies the outcome probability by exactly (1 + relative_effect)."""
    p_ref = sigmoid(beta0)
    p_target = p_ref * (1 + EXPERIENCE_THRESHOLD_RELATIVE_EFFECT)
    p_target = np.clip(p_target, 1e-9, 1 - 1e-9) 
    eta_target = np.log(p_target / (1 - p_target))
    return eta_target - beta0


def simulate_population_average_Y(beta0, n_ref, seed, tc_moments):
    """Simulate the FAIR reference population (g_female=0, but with active
    occupation moderation and the fixed NB effect) and return
    the empirical mean of Y for a given beta0 (with jointly-solved threshold coef)."""
    df = sample_population(n=n_ref, seed=seed)
    eta_base = compute_base_linear_predictor(df, tc_moments)

    beta_thr = beta_exp_threshold_given_beta0(beta0)
    exp_dummy = (df["Experience_Years"].values >= EXPERIENCE_THRESHOLD_YEARS).astype(float)

    delta = compute_delta(df["Z"].values, df["Occupation_Female_Share"].values, g_female=0.0)

    eta = beta0 + eta_base + beta_thr * exp_dummy + delta
    p = sigmoid(eta)
    y = np.mean(p)
    return y


def calibrate_beta0(target_rate=None, n_ref=200_000, seed=12345, tol=1e-5, max_iter=60):
    """Bisection search for beta0 such that the population-average outcome
    probability over the FAIR reference population equals target_rate exactly."""
    if target_rate is None:
        target_rate = Y_BASELINE_RATE_NOMINAL

    tc_moments = _compute_technical_communication_moments()

    lo, hi = -8.0, 2.0
    f_lo = simulate_population_average_Y(lo, n_ref, seed, tc_moments) - target_rate
    f_hi = simulate_population_average_Y(hi, n_ref, seed, tc_moments) - target_rate
    assert f_lo < 0 < f_hi, f"Bisection bracket invalid: f_lo={f_lo}, f_hi={f_hi}"

    history = []
    for i in range(max_iter):
        mid = (lo + hi) / 2
        f_mid = simulate_population_average_Y(mid, n_ref, seed, tc_moments) - target_rate
        history.append((mid, f_mid + target_rate))
        if abs(f_mid) < tol:
            break
        if f_mid < 0:
            lo = mid
        else:
            hi = mid

    beta0_final = mid
    beta_thr_final = beta_exp_threshold_given_beta0(beta0_final)
    return {
        "beta0": beta0_final,
        "beta_exp_threshold": beta_thr_final,
        "achieved_rate": history[-1][1],
        "target_rate": target_rate,
        "n_iterations": len(history),
        "history": history,
        "tc_moments": tc_moments,
    }

**Running the joint calibration** (bisection over β₀, ~15-20 iterations,
each evaluated on a 200,000-observation simulated reference population):

In [ ]:
cal = calibrate_beta0(n_ref=200_000, seed=12345)

print(f"beta0                = {cal['beta0']:.6f}")
print(f"beta_exp_threshold   = {cal['beta_exp_threshold']:.6f}")
print(f"target rate          = {cal['target_rate']*100:.4f}%")
print(f"achieved rate        = {cal['achieved_rate']*100:.4f}%")
print(f"converged in          {cal['n_iterations']} iterations")

naive_beta0 = np.log(Y_BASELINE_RATE_NOMINAL / (1 - Y_BASELINE_RATE_NOMINAL))
print(f"\n(naive, UNCALIBRATED logit(0.123) would have been: {naive_beta0:.4f}")
print(f" — the {abs(cal['beta0']-naive_beta0):.2f} gap demonstrates the Jensen's-inequality")
print(f" correction described above is not a negligible adjustment)")

In [ ]:
p_ref = sigmoid(cal['beta0'])
p_thr = sigmoid(cal['beta0'] + cal['beta_exp_threshold'])
print(f"Threshold effect check at reference point:")
print(f"  P(Y=1 | Experience<4)  = {p_ref*100:.4f}%")
print(f"  P(Y=1 | Experience>=4) = {p_thr*100:.4f}%")
print(f"  relative increase = {(p_thr/p_ref - 1)*100:.4f}%  (target: 11.5%)")

## 5. Dataset Generation: 11-Point Sweep

11 datasets at g_female = 0%, 10%, ..., 100% of Pisanelli's (2022) own reported
upper anchor (69% relative odds reduction), N=300,000 each, using fixed integer
seeds.


In [ ]:
def generate_dataset(g_female, n, seed, calibration):
    """Generate one complete dataset at a given g_female sweep level."""
    df = sample_population(n=n, seed=seed)

    tc_moments = calibration["tc_moments"]
    beta0 = calibration["beta0"]
    beta_thr = calibration["beta_exp_threshold"]

    eta_base = compute_base_linear_predictor(df, tc_moments)
    exp_dummy = (df["Experience_Years"].values >= EXPERIENCE_THRESHOLD_YEARS).astype(float)
    delta = compute_delta(df["Z"].values, df["Occupation_Female_Share"].values, g_female=g_female)

    eta = beta0 + eta_base + beta_thr * exp_dummy + delta
    p_true = sigmoid(eta)

    # second, independent random draw for the Bernoulli outcome itself
    rng_y = np.random.default_rng(seed + 500_000) 
    y = rng_y.binomial(1, p_true)

    df["Y"] = y
    df["P_true"] = p_true
    df["delta_applied"] = delta
    return df


def generate_all_sweep_datasets(calibration, n=None, verbose=True):
    """Generate all N_SWEEP_STEPS datasets, keyed by sweep index (0-10)."""
    if n is None:
        n = N_PER_DATASET

    datasets = {}
    for i, g in enumerate(G_FEMALE_SWEEP):
        seed = DATASET_SEEDS[i]
        df = generate_dataset(g_female=g, n=n, seed=seed, calibration=calibration)
        datasets[i] = {"g_female": g, "seed": seed, "df": df}
        if verbose:
            y_rate = df["Y"].mean()
            print(f"  [{i:2d}/10] g_female={g:.4f} ({g/G_FEMALE_MAX*100:5.1f}% of anchor), "
                  f"seed={seed}, N={len(df):,}, overall Y rate={y_rate*100:.3f}%")
    return datasets

In [ ]:
datasets = generate_all_sweep_datasets(cal, n=N_PER_DATASET, verbose=True)

**Summary table across the sweep:**

In [ ]:
rows = []
for i, d in datasets.items():
    df = d["df"]
    g = d["g_female"]
    rows.append({
        "step": i, "g_female": round(g, 3), "%_of_anchor": f"{g/G_FEMALE_MAX*100:.0f}%",
        "Y_overall": df["Y"].mean(), "Y_male": df.loc[df.Z=="Male","Y"].mean(),
        "Y_female": df.loc[df.Z=="Female","Y"].mean(), "Y_nonbinary": df.loc[df.Z=="Non-binary","Y"].mean(),
    })
summary = pd.DataFrame(rows)
summary["DI_female_vs_male"] = summary["Y_female"] / summary["Y_male"]
summary["DI_nonbinary_vs_male"] = summary["Y_nonbinary"] / summary["Y_male"]
summary

## 6. Verification

Five checks confirming the implementation reproduces its literature-derived
targets: (1) coefficient recovery via logistic regression, (2) truncated-normal
correction accuracy, (3) non-binary interpolation accuracy, (4/4b) non-binary
effect constancy across the sweep,(5) δ sanity against the empirical Female-vs-Male gap.


In [ ]:
from sklearn.linear_model import LogisticRegression

df_fair = datasets[0]["df"].copy()
tc = cal["tc_moments"]

edu_z = (df_fair["Education_Level"] - EDUCATION_ORDINAL_MEAN) / EDUCATION_ORDINAL_SD
age_z = (df_fair["Age_Bracket"] - AGE_ORDINAL_MEAN) / AGE_ORDINAL_SD
tech_z = (df_fair["Technical_Score"] - tc["technical_mean"]) / tc["technical_sd"]
comm_z = (df_fair["Communication_Score"] - tc["communication_mean"]) / tc["communication_sd"]
exp_dummy = (df_fair["Experience_Years"] >= EXPERIENCE_THRESHOLD_YEARS).astype(int)

X = pd.DataFrame({
    "Education_z": edu_z, "Technical_z": tech_z, "Communication_z": comm_z, "Age_z": age_z,
    "Motivation": df_fair["Motivation"], "Leadership": df_fair["Leadership"],
    "Employment_Gap": df_fair["Employment_Gap"], "Exp_Threshold": exp_dummy,
})
y = df_fair["Y"]
clf = LogisticRegression(max_iter=1000).fit(X, y)

targets = {
    "Education_z": BETA_EDUCATION, "Technical_z": BETA_TECHNICAL, "Communication_z": BETA_COMMUNICATION,
    "Age_z": BETA_AGE, "Motivation": BETA_MOTIVATION, "Leadership": BETA_LEADERSHIP,
    "Employment_Gap": BETA_EMPLOYMENT_GAP, "Exp_Threshold": cal["beta_exp_threshold"],
}
print("CHECK 1: Coefficient recovery (fair dataset)")
print(f"{'Variable':<18}{'Recovered':>12}{'Target':>12}{'AbsDiff':>10}")
for col, coef in zip(X.columns, clf.coef_[0]):
    t = targets[col]
    print(f"{col:<18}{coef:>12.4f}{t:>12.4f}{abs(coef-t):>10.4f}")
print(f"{'Intercept':<18}{clf.intercept_[0]:>12.4f}{cal['beta0']:>12.4f}{abs(clf.intercept_[0]-cal['beta0']):>10.4f}")

In [ ]:
print("CHECK 2: Truncated-normal correction accuracy")
for col, means, sds in [("Experience_Years", EXPERIENCE_MEAN, EXPERIENCE_SD),
                        ("Technical_Score", TECHNICAL_MEAN, TECHNICAL_SD),
                        ("Communication_Score", COMMUNICATION_MEAN, COMMUNICATION_SD)]:
    print(f"\n  {col}:")
    for z in ["Male", "Female", "Non-binary"]:
        sub = df_fair.loc[df_fair.Z == z, col]
        print(f"    {z:<12} mean={sub.mean():7.3f} (target {means[z]:7.3f})   sd={sub.std():6.3f} (target {sds[z]:6.3f})")

In [ ]:
print("CHECK 3: Non-binary interpolation accuracy")
edu_nb = df_fair.loc[df_fair.Z == "Non-binary", "Education_Level"]
print(f"  Education NB: HS-or-less = {(edu_nb<=3).mean()*100:.2f}% (target 48.00%)")
age_nb = df_fair.loc[df_fair.Z == "Non-binary", "Age_Bracket"]
print(f"  Age NB: <35 = {(age_nb<=3).mean()*100:.2f}% (target 87.00%)")

In [ ]:
print("CHECK 4: Non-binary effect — empirical constancy across the sweep")
nb_rates = [datasets[i]["df"].loc[datasets[i]["df"].Z=="Non-binary","Y"].mean() for i in range(11)]
g_vals = [datasets[i]["g_female"] for i in range(11)]
print(f"  NB Y-rates: {[f'{r*100:.2f}%' for r in nb_rates]}")
print(f"  Correlation(g_female, NB_rate) = {np.corrcoef(g_vals, nb_rates)[0,1]:.3f}")
print(f"  (with only 11 points and ~4,500 obs/group, noise-driven correlations up to")
print(f"   ~0.4-0.6 are expected even with zero true effect — see structural check below)")

print("\nCHECK 4b: Non-binary effect — STRUCTURAL constancy (deterministic, not sampled)")
all_nb_deltas = np.concatenate([
    datasets[i]["df"].loc[datasets[i]["df"].Z=="Non-binary","delta_applied"].unique()
    for i in range(11)
])
print(f"  Unique delta_applied values for NB across ALL 11 datasets: {np.unique(all_nb_deltas)}")
print(f"  Expected: ln(1-0.18) = {np.log(1-0.18):.10f}")
print(f"  => Confirms exactly (not just statistically) that NB's delta is independent of g_female.")

In [ ]:
print("CHECK 5: delta sanity — empirical Female-vs-Male gap tracks g_female")
for i in [0, 5, 10]:
    df = datasets[i]["df"]
    g = datasets[i]["g_female"]
    y_m, y_f = df.loc[df.Z=="Male","Y"].mean(), df.loc[df.Z=="Female","Y"].mean()
    print(f"  step {i}: g_input={g:.3f}  ->  empirical relative F-vs-M reduction={1-(y_f/y_m):.3f}")
print("  (step 0's non-zero reduction reflects the intended indirect X-channel;")
print("   later steps increasingly reflect the direct delta channel as well)")

## 7. Saving Datasets to CSV

Exports all 11 sweep datasets as CSV files plus a metadata file recording the
calibrated parameters (β₀, β_ExpThreshold) and the sweep specification for
reproducibility and use in Chapter 4.


In [ ]:
import os
import json

output_dir = "dgp_datasets"
os.makedirs(output_dir, exist_ok=True)

for i, d in datasets.items():
    g = d["g_female"]
    fname = f"{output_dir}/dataset_g{g:.4f}_step{i:02d}.csv"
    d["df"].to_csv(fname, index=False)
    print(f"Saved {fname}  ({len(d['df']):,} rows)")

metadata = {
    "beta0": cal["beta0"],
    "beta_exp_threshold": cal["beta_exp_threshold"],
    "tc_moments": cal["tc_moments"],
    "n_per_dataset": N_PER_DATASET,
    "g_female_sweep": list(G_FEMALE_SWEEP),
    "g_female_max_anchor": G_FEMALE_MAX,
    "g_nonbinary_fixed": G_NONBINARY_FIXED,
    "gamma_moderation": GAMMA_MODERATION,
    "dataset_seeds": DATASET_SEEDS,
}
with open(f"{output_dir}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(f"\nSaved {output_dir}/metadata.json")

## 8. Experiment Setup

Implements the experiment from Chapter 4.

A Logistic Regression baseline is trained, then three bias-correction
methods are applied, one per family (Table 2.2):

1. **Pre-processing:** Reweighing (Kamiran & Calders, 2012) — Male/Female only
2. **In-processing:** GridSearch (Agarwal et al., 2018), `EqualizedOdds()`,
   `grid_size=50` — all three Z levels
3. **Post-processing:** ThresholdOptimizer (Hardt, Price & Srebro, 2016),
   `constraints="equalized_odds"`, `objective="accuracy_score"` — all three
   Z levels

All four use **25 repeated train/test splits** (seeds 100–124) per dataset,
reporting mean ± SD.

### Setup Check

Verifies `datasets` and `cal` are already in memory from the DGP section
above, and sets up the output directory for this part's results. Does
not regenerate the datasets.


In [ ]:
import pickle, time, os, warnings, logging
warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)

# Sanity check: these must already exist from the DGP section above.
# If this fails, run the full notebook from the top.
assert "datasets" in dir(), (
    "`datasets` not found — run the DGP sections above first "
    "(this part reuses them rather than regenerating)."
)
assert "cal" in dir(), "`cal` not found — run the DGP calibration cell above first."
assert len(datasets) == 11, f"Expected 11 sweep datasets, found {len(datasets)}."

print("Reusing existing `datasets` and `cal` from the DGP section above.")
print(f"  {len(datasets)} datasets, each with {len(datasets[0]['df']):,} rows")
print(f"  beta0={cal['beta0']:.6f}, beta_exp_threshold={cal['beta_exp_threshold']:.6f}")

# Output directory for this part's incremental results
OUTPUT_DIR = "results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 9. Feature Matrix Construction

Builds the classifier feature matrix per the Chapter 4 design decisions:

- **Education_Level:** ordinal integer, used as-is
- **Experience_Years:** raw continuous
- **Technical_Score, Communication_Score:** continuous, used as-is
- **Occupation:** one-hot encoded, 21 categories → 20 dummy columns. Reference
  category: **"Other"**, deliberately chosen for interpretability
- **Age_Bracket:** ordinal integer, used as-is
- **Motivation, Leadership, Employment_Gap:** already binary 0/1
- **Excluded:** Occupation_Female_Share, P_true, delta_applied
- **Z (sensitive attribute):** kept separate, passed to the correction
  methods, not included as a classifier input feature


In [ ]:
def build_feature_matrix(df):
    """
    Build the classifier feature matrix per Chapter 4 design decisions.
    See the markdown cell above for the full column-by-column justification.
    Reference category for Occupation one-hot: "Other" (deliberately chosen
    for interpretability; see Section 4.1).
    """
    occ_dummies = pd.get_dummies(df["Occupation"], prefix="Occ")
    occ_dummies = occ_dummies.drop(columns=["Occ_Other"])  # deliberate reference category

    X = pd.DataFrame({
        "Education_Level": df["Education_Level"].values,
        "Experience_Years": df["Experience_Years"].values,
        "Technical_Score": df["Technical_Score"].values,
        "Communication_Score": df["Communication_Score"].values,
        "Age_Bracket": df["Age_Bracket"].values,
        "Motivation": df["Motivation"].values,
        "Leadership": df["Leadership"].values,
        "Employment_Gap": df["Employment_Gap"].values,
    })
    X = pd.concat([X, occ_dummies.astype(int)], axis=1)

    y = df["Y"].values
    z = df["Z"].values
    return X, y, z


# Quick check on the fair (g=0) dataset
X0, y0, z0 = build_feature_matrix(datasets[0]["df"])
print("Feature matrix shape:", X0.shape)
print("Columns:", list(X0.columns))
print("Z counts:", dict(zip(*np.unique(z0, return_counts=True))))
print("Y rate:", y0.mean())


## 10. Baseline Classifier (Uncorrected)

Logistic Regression, no fairness correction. Classification threshold is
**calibrated to the empirical training-set Y base rate** (selecting the
top-K% by predicted probability, K = training set's own observed Y rate)
rather than the default 0.5 cutoff (Feldman et al., 2015).

25 repeated train/test splits (seeds 100–124) per dataset; StandardScaler
fit on training data only.

In [ ]:
# Shared helper functions (used by Baseline, Reweighing, and reported
# alongside GridSearch / ThresholdOptimizer for consistency)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

N_REPEATS = 25
SEEDS = list(range(100, 100 + N_REPEATS))


def calibrated_threshold(y_proba_train, y_train_rate):
    """Cutoff so the predicted-positive rate on train matches the empirical
    Y base rate of that dataset (own methodological decision, Section 4.2;
    consistent with the calibration logic of Section 3.2.3)."""
    return np.quantile(y_proba_train, 1 - y_train_rate)


def disparate_impact_ratio(y_pred, z, ref_group="Male"):
    """Disparate Impact Ratio per group, relative to Male (Section 4.2b)."""
    rates = {g: y_pred[z == g].mean() for g in np.unique(z)}
    ref_rate = rates[ref_group]
    return {g: (rates[g] / ref_rate if ref_rate > 0 else np.nan) for g in rates}


def equal_opportunity_diff(y_true, y_pred, z, ref_group="Male"):
    """Equal Opportunity Difference (TPR gap) per group, relative to Male."""
    tprs = {}
    for g in np.unique(z):
        mask = (z == g) & (y_true == 1)
        tprs[g] = y_pred[mask].mean() if mask.sum() > 0 else np.nan
    ref_tpr = tprs[ref_group]
    return {g: (tprs[g] - ref_tpr) for g in tprs}, tprs


In [ ]:
# Baseline: 25 repeats x 11 datasets
t0 = time.time()
all_rows = []
for idx in sorted(datasets.keys()):
    dd = datasets[idx]
    df, g = dd["df"], dd["g_female"]
    X, y, z = build_feature_matrix(df)

    for seed in SEEDS:
        X_train, X_test, y_train, y_test, z_train, z_test = train_test_split(
            X, y, z, test_size=0.3, random_state=seed, stratify=z
        )
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        clf = LogisticRegression(max_iter=500)
        clf.fit(X_train_s, y_train)
        proba_train = clf.predict_proba(X_train_s)[:, 1]
        proba_test = clf.predict_proba(X_test_s)[:, 1]
        thresh = calibrated_threshold(proba_train, y_train.mean())
        y_pred = (proba_test >= thresh).astype(int)

        di = disparate_impact_ratio(y_pred, z_test)
        eod, tprs = equal_opportunity_diff(y_test, y_pred, z_test)

        all_rows.append({
            "g_female": g, "seed": seed,
            "DI_Female": di["Female"], "DI_NonBinary": di["Non-binary"],
            "EOD_Female": eod["Female"], "EOD_NonBinary": eod["Non-binary"],
            "Y_pred_rate": y_pred.mean(), "Y_true_rate": y_test.mean(),
        })
    print(f"  g={g:.3f} done ({time.time()-t0:.1f}s elapsed)", flush=True)

baseline_raw = pd.DataFrame(all_rows)
baseline_raw.to_pickle(f"{OUTPUT_DIR}/baseline_raw.pkl")

baseline_summary = baseline_raw.groupby("g_female").agg(
    DI_Female_mean=("DI_Female","mean"), DI_Female_sd=("DI_Female","std"),
    DI_NonBinary_mean=("DI_NonBinary","mean"), DI_NonBinary_sd=("DI_NonBinary","std"),
    EOD_Female_mean=("EOD_Female","mean"), EOD_Female_sd=("EOD_Female","std"),
    EOD_NonBinary_mean=("EOD_NonBinary","mean"), EOD_NonBinary_sd=("EOD_NonBinary","std"),
).reset_index()
baseline_summary.to_pickle(f"{OUTPUT_DIR}/baseline_summary.pkl")
baseline_summary.to_csv(f"{OUTPUT_DIR}/baseline_summary.csv", index=False)

print(f"\nBASELINE — {N_REPEATS} repeats, total runtime {time.time()-t0:.1f}s")
print(baseline_summary.to_string(index=False))

## 11. Reweighing (Pre-processing)

AIF360's implementation of Kamiran & Calders (2012). Applied to the
**Male/Female comparison only** since the method is structurally restricted to a
binary sensitive attribute. Non-binary is therefore evaluated only under
GridSearch and ThresholdOptimizer below (Section 12/13).

In [ ]:
# Reweighing: 25 repeats x 11 datasets, Male/Female only
from aif360.algorithms.preprocessing import Reweighing
from aif360.datasets import BinaryLabelDataset


def get_reweighing_sample_weights(y_train, z_train):
    """AIF360 Reweighing (Kamiran & Calders, 2012), Male/Female rows only —
    method structurally restricted to a binary sensitive attribute."""
    aif_df = pd.DataFrame({"protected": (z_train == "Female").astype(int), "label": y_train})
    bld = BinaryLabelDataset(df=aif_df, label_names=["label"],
                              protected_attribute_names=["protected"],
                              favorable_label=1, unfavorable_label=0)
    rw = Reweighing(unprivileged_groups=[{"protected": 1}],
                     privileged_groups=[{"protected": 0}])
    bld_transf = rw.fit_transform(bld)
    return bld_transf.instance_weights


def di_binary(y_pred, z, ref="Male", other="Female"):
    r_ref = y_pred[z == ref].mean()
    r_other = y_pred[z == other].mean()
    return r_other / r_ref if r_ref > 0 else np.nan


def eod_binary(y_true, y_pred, z, ref="Male", other="Female"):
    m_ref = (z == ref) & (y_true == 1)
    m_other = (z == other) & (y_true == 1)
    return y_pred[m_other].mean() - y_pred[m_ref].mean()


t0 = time.time()
all_rows = []
for idx in sorted(datasets.keys()):
    dd = datasets[idx]
    df_full, g = dd["df"], dd["g_female"]
    df = df_full[df_full["Z"].isin(["Male", "Female"])].reset_index(drop=True)
    X, y, z = build_feature_matrix(df)

    for seed in SEEDS:
        X_train, X_test, y_train, y_test, z_train, z_test = train_test_split(
            X, y, z, test_size=0.3, random_state=seed, stratify=z
        )
        sw = get_reweighing_sample_weights(y_train, z_train)

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        clf = LogisticRegression(max_iter=500)
        clf.fit(X_train_s, y_train, sample_weight=sw)

        proba_train = clf.predict_proba(X_train_s)[:, 1]
        proba_test = clf.predict_proba(X_test_s)[:, 1]
        thresh = np.quantile(proba_train, 1 - y_train.mean())
        y_pred = (proba_test >= thresh).astype(int)

        di = di_binary(y_pred, z_test)
        eod = eod_binary(y_test, y_pred, z_test)
        all_rows.append({"g_female": g, "seed": seed, "DI_Female": di, "EOD_Female": eod})
    print(f"  g={g:.3f} done ({time.time()-t0:.1f}s elapsed)", flush=True)

reweighing_raw = pd.DataFrame(all_rows)
reweighing_raw.to_pickle(f"{OUTPUT_DIR}/reweighing_raw.pkl")

reweighing_summary = reweighing_raw.groupby("g_female").agg(
    DI_Female_mean=("DI_Female","mean"), DI_Female_sd=("DI_Female","std"),
    EOD_Female_mean=("EOD_Female","mean"), EOD_Female_sd=("EOD_Female","std"),
).reset_index()
reweighing_summary.to_pickle(f"{OUTPUT_DIR}/reweighing_summary.pkl")
reweighing_summary.to_csv(f"{OUTPUT_DIR}/reweighing_summary.csv", index=False)

print(f"\nREWEIGHING — {N_REPEATS} repeats, total runtime {time.time()-t0:.1f}s")
print(reweighing_summary.to_string(index=False))

## 12. GridSearch (In-processing)

fairlearn's `GridSearch` with `EqualizedOdds()` constraint, `grid_size=50`. 
All three Z levels supported natively.

In [ ]:
# GridSearch: 25 repeats x 11 datasets, grid_size=50, checkpointed
from fairlearn.reductions import GridSearch, EqualizedOdds

GRID_SIZE = 50

t0 = time.time()
all_rows = []
for idx in sorted(datasets.keys()):
    dd = datasets[idx]
    df, g = dd["df"], dd["g_female"]
    X, y, z = build_feature_matrix(df)

    for seed in SEEDS:
        X_train, X_test, y_train, y_test, z_train, z_test = train_test_split(
            X, y, z, test_size=0.3, random_state=seed, stratify=z
        )
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        gs = GridSearch(estimator=LogisticRegression(max_iter=500),
                         constraints=EqualizedOdds(), grid_size=GRID_SIZE)
        gs.fit(X_train_s, y_train, sensitive_features=z_train)
        y_pred = gs.predict(X_test_s)

        di = disparate_impact_ratio(y_pred, z_test)
        eod, tprs = equal_opportunity_diff(y_test, y_pred, z_test)

        all_rows.append({
            "g_female": g, "seed": seed,
            "DI_Female": di["Female"], "DI_NonBinary": di["Non-binary"],
            "EOD_Female": eod["Female"], "EOD_NonBinary": eod["Non-binary"],
            "Y_pred_rate": y_pred.mean(),
        })
    elapsed = time.time() - t0
    print(f"  g={g:.3f} done ({elapsed:.1f}s elapsed, dataset {idx+1}/11)", flush=True)
    # Checkpoint after every dataset, to allow safe resumption if interrupted
    pd.DataFrame(all_rows).to_pickle(f"{OUTPUT_DIR}/gridsearch_checkpoint.pkl")
    with open(f"{OUTPUT_DIR}/gridsearch_progress.txt", "w") as f:
        f.write(f"{idx+1}/11 datasets done, {elapsed:.1f}s elapsed\n")

gridsearch_raw = pd.DataFrame(all_rows)
gridsearch_raw.to_pickle(f"{OUTPUT_DIR}/gridsearch_raw.pkl")

gridsearch_summary = gridsearch_raw.groupby("g_female").agg(
    DI_Female_mean=("DI_Female","mean"), DI_Female_sd=("DI_Female","std"),
    DI_NonBinary_mean=("DI_NonBinary","mean"), DI_NonBinary_sd=("DI_NonBinary","std"),
    EOD_Female_mean=("EOD_Female","mean"), EOD_Female_sd=("EOD_Female","std"),
    EOD_NonBinary_mean=("EOD_NonBinary","mean"), EOD_NonBinary_sd=("EOD_NonBinary","std"),
).reset_index()
gridsearch_summary.to_pickle(f"{OUTPUT_DIR}/gridsearch_summary.pkl")
gridsearch_summary.to_csv(f"{OUTPUT_DIR}/gridsearch_summary.csv", index=False)

print(f"\nGRIDSEARCH — {N_REPEATS} repeats, grid_size={GRID_SIZE}, total runtime {time.time()-t0:.1f}s")
print(gridsearch_summary.to_string(index=False))

### 12a. GridSearch — Robustness Check at fairlearn's Default grid_size

Re-runs the sweep at **grid_size=10 — fairlearn's documented default**
(confirmed from `GridSearch.__init__`), to check whether the instability
seen at grid_size=50 away from g=0 is a grid-resolution artifact or a more
fundamental property of 3-group EqualizedOdds. 

In [ ]:
# GridSearch robustness check: grid_size=10 (fairlearn's actual default),
# 25 repeats x 11 datasets, checkpointed
from fairlearn.reductions import GridSearch, EqualizedOdds
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd, numpy as np, time

assert "datasets" in dir(), "Run the DGP + Chapter 4 sections above first."

GRID_SIZE_COARSE = 10 

t0 = time.time()
all_rows = []
for idx in sorted(datasets.keys()):
    dd = datasets[idx]
    df, g = dd["df"], dd["g_female"]
    X, y, z = build_feature_matrix(df)

    for seed in SEEDS:  # same 25 seeds (100-124) as the grid_size=50 run, for direct comparability
        X_train, X_test, y_train, y_test, z_train, z_test = train_test_split(
            X, y, z, test_size=0.3, random_state=seed, stratify=z
        )
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        gs = GridSearch(estimator=LogisticRegression(max_iter=500),
                         constraints=EqualizedOdds(), grid_size=GRID_SIZE_COARSE)
        gs.fit(X_train_s, y_train, sensitive_features=z_train)
        y_pred = gs.predict(X_test_s)

        di = disparate_impact_ratio(y_pred, z_test)
        eod, tprs = equal_opportunity_diff(y_test, y_pred, z_test)

        all_rows.append({
            "g_female": g, "seed": seed,
            "DI_Female": di["Female"], "DI_NonBinary": di["Non-binary"],
            "EOD_Female": eod["Female"], "EOD_NonBinary": eod["Non-binary"],
        })
    elapsed = time.time() - t0
    print(f"  g={g:.3f} done ({elapsed:.1f}s elapsed, dataset {idx+1}/11)", flush=True)
    pd.DataFrame(all_rows).to_pickle(f"{OUTPUT_DIR}/gridsearch_coarse_checkpoint.pkl")

gridsearch_coarse_raw = pd.DataFrame(all_rows)
gridsearch_coarse_raw.to_pickle(f"{OUTPUT_DIR}/gridsearch_coarse_raw.pkl")

gridsearch_coarse_summary = gridsearch_coarse_raw.groupby("g_female").agg(
    DI_Female_mean=("DI_Female","mean"), DI_Female_sd=("DI_Female","std"),
    DI_NonBinary_mean=("DI_NonBinary","mean"), DI_NonBinary_sd=("DI_NonBinary","std"),
    EOD_Female_mean=("EOD_Female","mean"), EOD_Female_sd=("EOD_Female","std"),
    EOD_NonBinary_mean=("EOD_NonBinary","mean"), EOD_NonBinary_sd=("EOD_NonBinary","std"),
).reset_index()
gridsearch_coarse_summary.to_pickle(f"{OUTPUT_DIR}/gridsearch_coarse_summary.pkl")
gridsearch_coarse_summary.to_csv(f"{OUTPUT_DIR}/gridsearch_coarse_summary.csv", index=False)

print(f"\nGRIDSEARCH (grid_size={GRID_SIZE_COARSE}) — {len(SEEDS)} repeats, total runtime {time.time()-t0:.1f}s")
print(gridsearch_coarse_summary.to_string(index=False))

**Direct comparison, grid_size=10 vs. grid_size=50 (DI_Female SD by g_female):**

Run the cell below after both sweeps have completed to see the side-by-side
SD comparison that supports the "not primarily a grid
resolution artifact" claim above.

In [ ]:
# Side-by-side comparison: grid_size=10 vs grid_size=50 stability
grid_compare = pd.DataFrame({
    "g_female": gridsearch_summary["g_female"],
    "DI_Female_SD_grid10": gridsearch_coarse_summary["DI_Female_sd"].values,
    "DI_Female_SD_grid50": gridsearch_summary["DI_Female_sd"].values,
})
grid_compare["grid50_more_stable"] = grid_compare["DI_Female_SD_grid50"] < grid_compare["DI_Female_SD_grid10"]
grid_compare.to_csv(f"{OUTPUT_DIR}/grid_size_comparison.csv", index=False)

print(grid_compare.to_string(index=False))
print(f"\ngrid_size=50 more stable than grid_size=10 in {grid_compare['grid50_more_stable'].sum()}/11 points")

## 13. Threshold Optimization (Post-processing)

fairlearn's `ThresholdOptimizer`, Hardt, Price & Srebro (2016).
`constraints="equalized_odds"`, `objective="accuracy_score"`.

**Note:** unlike the other three methods, there is no free threshold-
calibration step. The resulting predicted-positive rate (~4%) is far
below the 12.3% used elsewhere, since `objective="accuracy_score"` doesn't
target a specific selection rate.

In [ ]:
# ThresholdOptimizer: 25 repeats x 11 datasets
from fairlearn.postprocessing import ThresholdOptimizer

t0 = time.time()
all_rows = []
for idx in sorted(datasets.keys()):
    dd = datasets[idx]
    df, g = dd["df"], dd["g_female"]
    X, y, z = build_feature_matrix(df)

    for seed in SEEDS:
        X_train, X_test, y_train, y_test, z_train, z_test = train_test_split(
            X, y, z, test_size=0.3, random_state=seed, stratify=z
        )
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        base_clf = LogisticRegression(max_iter=500)
        base_clf.fit(X_train_s, y_train)

        to = ThresholdOptimizer(estimator=base_clf, constraints="equalized_odds",
                                 objective="accuracy_score", predict_method="predict_proba",
                                 prefit=True)
        to.fit(X_train_s, y_train, sensitive_features=z_train)
        y_pred = to.predict(X_test_s, sensitive_features=z_test, random_state=seed)

        di = disparate_impact_ratio(y_pred, z_test)
        eod, tprs = equal_opportunity_diff(y_test, y_pred, z_test)

        all_rows.append({
            "g_female": g, "seed": seed,
            "DI_Female": di["Female"], "DI_NonBinary": di["Non-binary"],
            "EOD_Female": eod["Female"], "EOD_NonBinary": eod["Non-binary"],
            "Y_pred_rate": y_pred.mean(),
        })
    print(f"  g={g:.3f} done ({time.time()-t0:.1f}s elapsed)", flush=True)

to_raw = pd.DataFrame(all_rows)
to_raw.to_pickle(f"{OUTPUT_DIR}/thresholdopt_raw.pkl")

to_summary = to_raw.groupby("g_female").agg(
    DI_Female_mean=("DI_Female","mean"), DI_Female_sd=("DI_Female","std"),
    DI_NonBinary_mean=("DI_NonBinary","mean"), DI_NonBinary_sd=("DI_NonBinary","std"),
    EOD_Female_mean=("EOD_Female","mean"), EOD_Female_sd=("EOD_Female","std"),
    EOD_NonBinary_mean=("EOD_NonBinary","mean"), EOD_NonBinary_sd=("EOD_NonBinary","std"),
    Y_pred_rate_mean=("Y_pred_rate","mean"),
).reset_index()
to_summary.to_pickle(f"{OUTPUT_DIR}/thresholdopt_summary.pkl")
to_summary.to_csv(f"{OUTPUT_DIR}/thresholdopt_summary.csv", index=False)

print(f"\nTHRESHOLD OPTIMIZATION — {N_REPEATS} repeats, total runtime {time.time()-t0:.1f}s")
print(to_summary.to_string(index=False))

## 14. Combined Comparison Table

Combines all four methods (Baseline, Reweighing, GridSearch,
ThresholdOptimizer) into one comparison table across the g_female sweep.

In [ ]:
# Combined comparison table across all four methods
comparison_rows = []
for g in baseline_summary["g_female"]:
    b = baseline_summary[baseline_summary.g_female == g].iloc[0]
    r = reweighing_summary[reweighing_summary.g_female == g].iloc[0]
    gs = gridsearch_summary[gridsearch_summary.g_female == g].iloc[0]
    t = to_summary[to_summary.g_female == g].iloc[0]

    comparison_rows.append({
        "g_female": g,
        "DI_Female_Baseline": f"{b.DI_Female_mean:.3f} ({b.DI_Female_sd:.3f})",
        "DI_Female_Reweighing": f"{r.DI_Female_mean:.3f} ({r.DI_Female_sd:.3f})",
        "DI_Female_GridSearch": f"{gs.DI_Female_mean:.3f} ({gs.DI_Female_sd:.3f})",
        "DI_Female_ThresholdOpt": f"{t.DI_Female_mean:.3f} ({t.DI_Female_sd:.3f})",
        "DI_NonBinary_Baseline": f"{b.DI_NonBinary_mean:.3f} ({b.DI_NonBinary_sd:.3f})",
        "DI_NonBinary_Reweighing": "N/A — method structurally binary",
        "DI_NonBinary_GridSearch": f"{gs.DI_NonBinary_mean:.3f} ({gs.DI_NonBinary_sd:.3f})",
        "DI_NonBinary_ThresholdOpt": f"{t.DI_NonBinary_mean:.3f} ({t.DI_NonBinary_sd:.3f})",
    })

comparison_table = pd.DataFrame(comparison_rows)
comparison_table.to_csv(f"{OUTPUT_DIR}/comparison_table.csv", index=False)
comparison_table.to_pickle(f"{OUTPUT_DIR}/comparison_table.pkl")

print("COMBINED COMPARISON TABLE (mean (SD) format)")
print(comparison_table.to_string(index=False))
print(f"\nAll results saved under {OUTPUT_DIR}/")
print("Files:", sorted(os.listdir(OUTPUT_DIR)))

## 15. Accuracy - Supplementary Fairness-Accuracy Trade-off

Adds `accuracy_score` for all four methods, since fairness metrics alone
don't show the trade-off.

**5 repeated splits (seeds 100–104), not 25.** Accuracy has much lower
variance than DI, so fewer repeats suffice.

In [ ]:
# Supplementary accuracy computation — 5 repeats, all 4 methods, all 11 datasets
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from fairlearn.reductions import GridSearch, EqualizedOdds
from fairlearn.postprocessing import ThresholdOptimizer
from aif360.algorithms.preprocessing import Reweighing
from aif360.datasets import BinaryLabelDataset
import time

assert "datasets" in dir(), "Run the DGP + Chapter 4 sections above first."

ACC_SEEDS = list(range(100, 105)) 
GRID_SIZE = 50

def get_reweighing_sample_weights(y_train, z_train):
    aif_df = pd.DataFrame({"protected": (z_train == "Female").astype(int), "label": y_train})
    bld = BinaryLabelDataset(df=aif_df, label_names=["label"],
                              protected_attribute_names=["protected"],
                              favorable_label=1, unfavorable_label=0)
    rw = Reweighing(unprivileged_groups=[{"protected": 1}], privileged_groups=[{"protected": 0}])
    return rw.fit_transform(bld).instance_weights

t0 = time.time()
acc_rows = []
for idx in sorted(datasets.keys()):
    dd = datasets[idx]
    df, g = dd["df"], dd["g_female"]
    X, y, z = build_feature_matrix(df)

    for seed in ACC_SEEDS:
        X_train, X_test, y_train, y_test, z_train, z_test = train_test_split(
            X, y, z, test_size=0.3, random_state=seed, stratify=z
        )
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        row = {"g_female": g, "seed": seed}

        # --- Baseline ---
        clf = LogisticRegression(max_iter=500)
        clf.fit(X_train_s, y_train)
        proba_train = clf.predict_proba(X_train_s)[:, 1]
        proba_test = clf.predict_proba(X_test_s)[:, 1]
        thresh = np.quantile(proba_train, 1 - y_train.mean())
        y_pred_base = (proba_test >= thresh).astype(int)
        row["Accuracy_Baseline"] = accuracy_score(y_test, y_pred_base)

        # --- Reweighing (Male/Female subset only, for a like-for-like comparison
        #     use the same M/F-only test subset for its accuracy figure) ---
        mf_mask_train = np.isin(z_train, ["Male", "Female"])
        mf_mask_test = np.isin(z_test, ["Male", "Female"])
        sw = get_reweighing_sample_weights(y_train[mf_mask_train], z_train[mf_mask_train])
        clf_rw = LogisticRegression(max_iter=500)
        clf_rw.fit(X_train_s[mf_mask_train], y_train[mf_mask_train], sample_weight=sw)
        proba_train_rw = clf_rw.predict_proba(X_train_s[mf_mask_train])[:, 1]
        proba_test_rw = clf_rw.predict_proba(X_test_s[mf_mask_test])[:, 1]
        thresh_rw = np.quantile(proba_train_rw, 1 - y_train[mf_mask_train].mean())
        y_pred_rw = (proba_test_rw >= thresh_rw).astype(int)
        row["Accuracy_Reweighing"] = accuracy_score(y_test[mf_mask_test], y_pred_rw)

        # --- GridSearch ---
        gs = GridSearch(estimator=LogisticRegression(max_iter=500),
                         constraints=EqualizedOdds(), grid_size=GRID_SIZE)
        gs.fit(X_train_s, y_train, sensitive_features=z_train)
        y_pred_gs = gs.predict(X_test_s)
        row["Accuracy_GridSearch"] = accuracy_score(y_test, y_pred_gs)

        # --- ThresholdOptimizer ---
        to = ThresholdOptimizer(estimator=clf, constraints="equalized_odds",
                                 objective="accuracy_score", predict_method="predict_proba",
                                 prefit=True)
        to.fit(X_train_s, y_train, sensitive_features=z_train)
        y_pred_to = to.predict(X_test_s, sensitive_features=z_test, random_state=seed)
        row["Accuracy_ThresholdOpt"] = accuracy_score(y_test, y_pred_to)

        acc_rows.append(row)
    print(f"  g={g:.3f} done ({time.time()-t0:.1f}s elapsed)", flush=True)

accuracy_raw = pd.DataFrame(acc_rows)
accuracy_raw.to_pickle(f"{OUTPUT_DIR}/accuracy_raw.pkl")

accuracy_summary = accuracy_raw.groupby("g_female").agg(
    Accuracy_Baseline_mean=("Accuracy_Baseline","mean"), Accuracy_Baseline_sd=("Accuracy_Baseline","std"),
    Accuracy_Reweighing_mean=("Accuracy_Reweighing","mean"), Accuracy_Reweighing_sd=("Accuracy_Reweighing","std"),
    Accuracy_GridSearch_mean=("Accuracy_GridSearch","mean"), Accuracy_GridSearch_sd=("Accuracy_GridSearch","std"),
    Accuracy_ThresholdOpt_mean=("Accuracy_ThresholdOpt","mean"), Accuracy_ThresholdOpt_sd=("Accuracy_ThresholdOpt","std"),
).reset_index()
accuracy_summary.to_pickle(f"{OUTPUT_DIR}/accuracy_summary.pkl")
accuracy_summary.to_csv(f"{OUTPUT_DIR}/accuracy_summary.csv", index=False)

print(f"\nACCURACY SUPPLEMENT — {len(ACC_SEEDS)} repeats, total runtime {time.time()-t0:.1f}s")
print(accuracy_summary.to_string(index=False))

**Note on the Reweighing accuracy figure:** computed and evaluated on
the Male/Female-only subset consistent with Reweighing itself being
Male/Female-only throughout this chapter. It is not directly comparable in
absolute terms to the other three methods' accuracy, only in relative trend across the sweep.


## 16. Visualizations

All plots use mean ± SD (shaded band) across the 25 repeats or 5 for accuray, not single-point estimates. Saved as PNG files under
`results/figures/` in addition to being shown inline.

1. Disparate Impact (Female, Non-binary) across the g_female sweep, one
   panel per group, all four methods overlaid
2. Equal Opportunity Difference (Female, Non-binary), same layout
3. Accuracy across the sweep, all four methods overlaid
4. Fairness-Accuracy trade-off: Accuracy vs. DI_Female, one line per
   method across the sweep
5. GridSearch stability check: DI_Female SD by g_female, grid_size=10 vs.
   grid_size=50

In [ ]:
# Plot setup
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

FIG_DIR = f"{OUTPUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

METHOD_COLORS = {
    "Baseline": "#7f7f7f",
    "Reweighing": "#1f77b4",
    "GridSearch": "#ff7f0e",
    "ThresholdOpt": "#2ca02c",
}
METHOD_LABELS = {
    "Baseline": "Baseline (uncorrected)",
    "Reweighing": "Reweighing (Pre)",
    "GridSearch": "GridSearch (In, grid=50)",
    "ThresholdOpt": "ThresholdOptimizer (Post)",
}

plt.rcParams.update({"figure.dpi": 100, "font.size": 10, "axes.grid": True,
                      "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})


def plot_metric_band(ax, g_vals, mean_vals, sd_vals, color, label):
    ax.plot(g_vals, mean_vals, marker="o", markersize=3, color=color, label=label, linewidth=1.6)
    ax.fill_between(g_vals, np.array(mean_vals) - np.array(sd_vals),
                     np.array(mean_vals) + np.array(sd_vals), color=color, alpha=0.15)

In [ ]:
# Figure 1: Disparate Impact across the sweep — Female & Non-binary
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

# Female panel
ax = axes[0]
plot_metric_band(ax, baseline_summary.g_female, baseline_summary.DI_Female_mean, baseline_summary.DI_Female_sd,
                  METHOD_COLORS["Baseline"], METHOD_LABELS["Baseline"])
plot_metric_band(ax, reweighing_summary.g_female, reweighing_summary.DI_Female_mean, reweighing_summary.DI_Female_sd,
                  METHOD_COLORS["Reweighing"], METHOD_LABELS["Reweighing"])
plot_metric_band(ax, gridsearch_summary.g_female, gridsearch_summary.DI_Female_mean, gridsearch_summary.DI_Female_sd,
                  METHOD_COLORS["GridSearch"], METHOD_LABELS["GridSearch"])
plot_metric_band(ax, to_summary.g_female, to_summary.DI_Female_mean, to_summary.DI_Female_sd,
                  METHOD_COLORS["ThresholdOpt"], METHOD_LABELS["ThresholdOpt"])
ax.axhline(1.0, color="black", linestyle="--", linewidth=1, alpha=0.5, label="Perfect parity")
ax.set_xlabel("g_female (sweep parameter)")
ax.set_ylabel("Disparate Impact Ratio")
ax.set_title("Female")

# Non-binary panel
ax = axes[1]
plot_metric_band(ax, baseline_summary.g_female, baseline_summary.DI_NonBinary_mean, baseline_summary.DI_NonBinary_sd,
                  METHOD_COLORS["Baseline"], METHOD_LABELS["Baseline"])
plot_metric_band(ax, gridsearch_summary.g_female, gridsearch_summary.DI_NonBinary_mean, gridsearch_summary.DI_NonBinary_sd,
                  METHOD_COLORS["GridSearch"], METHOD_LABELS["GridSearch"])
plot_metric_band(ax, to_summary.g_female, to_summary.DI_NonBinary_mean, to_summary.DI_NonBinary_sd,
                  METHOD_COLORS["ThresholdOpt"], METHOD_LABELS["ThresholdOpt"])
ax.axhline(1.0, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.set_xlabel("g_female (sweep parameter)")
ax.set_title("Non-binary (no Reweighing — structurally binary; see 4.3)")
ax.legend(loc="upper right", fontsize=8)

fig.suptitle("Disparate Impact Ratio across the Discrimination Sweep (mean ± SD, 25 repeats)")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig1_disparate_impact.png", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 2: Equal Opportunity Difference across the sweep — Female & Non-binary
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

ax = axes[0]
plot_metric_band(ax, baseline_summary.g_female, baseline_summary.EOD_Female_mean, baseline_summary.EOD_Female_sd,
                  METHOD_COLORS["Baseline"], METHOD_LABELS["Baseline"])
plot_metric_band(ax, reweighing_summary.g_female, reweighing_summary.EOD_Female_mean, reweighing_summary.EOD_Female_sd,
                  METHOD_COLORS["Reweighing"], METHOD_LABELS["Reweighing"])
plot_metric_band(ax, gridsearch_summary.g_female, gridsearch_summary.EOD_Female_mean, gridsearch_summary.EOD_Female_sd,
                  METHOD_COLORS["GridSearch"], METHOD_LABELS["GridSearch"])
plot_metric_band(ax, to_summary.g_female, to_summary.EOD_Female_mean, to_summary.EOD_Female_sd,
                  METHOD_COLORS["ThresholdOpt"], METHOD_LABELS["ThresholdOpt"])
ax.axhline(0.0, color="black", linestyle="--", linewidth=1, alpha=0.5, label="Perfect parity")
ax.set_xlabel("g_female (sweep parameter)")
ax.set_ylabel("Equal Opportunity Difference (TPR gap)")
ax.set_title("Female")

ax = axes[1]
plot_metric_band(ax, baseline_summary.g_female, baseline_summary.EOD_NonBinary_mean, baseline_summary.EOD_NonBinary_sd,
                  METHOD_COLORS["Baseline"], METHOD_LABELS["Baseline"])
plot_metric_band(ax, gridsearch_summary.g_female, gridsearch_summary.EOD_NonBinary_mean, gridsearch_summary.EOD_NonBinary_sd,
                  METHOD_COLORS["GridSearch"], METHOD_LABELS["GridSearch"])
plot_metric_band(ax, to_summary.g_female, to_summary.EOD_NonBinary_mean, to_summary.EOD_NonBinary_sd,
                  METHOD_COLORS["ThresholdOpt"], METHOD_LABELS["ThresholdOpt"])
ax.axhline(0.0, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.set_xlabel("g_female (sweep parameter)")
ax.set_title("Non-binary (no Reweighing — structurally binary; see 4.3)")
ax.legend(loc="upper right", fontsize=8)

fig.suptitle("Equal Opportunity Difference across the Discrimination Sweep (mean ± SD, 25 repeats)")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig2_equal_opportunity_diff.png", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 3: Accuracy across the sweep, all four methods
fig, ax = plt.subplots(figsize=(7, 4.5))
plot_metric_band(ax, accuracy_summary.g_female, accuracy_summary.Accuracy_Baseline_mean, accuracy_summary.Accuracy_Baseline_sd,
                  METHOD_COLORS["Baseline"], METHOD_LABELS["Baseline"])
plot_metric_band(ax, accuracy_summary.g_female, accuracy_summary.Accuracy_Reweighing_mean, accuracy_summary.Accuracy_Reweighing_sd,
                  METHOD_COLORS["Reweighing"], METHOD_LABELS["Reweighing"] + " (M/F subset)")
plot_metric_band(ax, accuracy_summary.g_female, accuracy_summary.Accuracy_GridSearch_mean, accuracy_summary.Accuracy_GridSearch_sd,
                  METHOD_COLORS["GridSearch"], METHOD_LABELS["GridSearch"])
plot_metric_band(ax, accuracy_summary.g_female, accuracy_summary.Accuracy_ThresholdOpt_mean, accuracy_summary.Accuracy_ThresholdOpt_sd,
                  METHOD_COLORS["ThresholdOpt"], METHOD_LABELS["ThresholdOpt"])
ax.set_xlabel("g_female (sweep parameter)")
ax.set_ylabel("Accuracy")
ax.set_title("Predictive Accuracy across the Discrimination Sweep (mean ± SD, 5 repeats)")
ax.legend(loc="best", fontsize=8)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig3_accuracy.png", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 4: Fairness-Accuracy trade-off — Accuracy vs. DI_Female,
# one line per method, ordered along the g_female sweep
fig, ax = plt.subplots(figsize=(7, 5.5))

merged = {
    "Baseline": baseline_summary.merge(accuracy_summary[["g_female","Accuracy_Baseline_mean"]], on="g_female"),
    "Reweighing": reweighing_summary.merge(accuracy_summary[["g_female","Accuracy_Reweighing_mean"]], on="g_female"),
    "GridSearch": gridsearch_summary.merge(accuracy_summary[["g_female","Accuracy_GridSearch_mean"]], on="g_female"),
    "ThresholdOpt": to_summary.merge(accuracy_summary[["g_female","Accuracy_ThresholdOpt_mean"]], on="g_female"),
}
acc_col = {"Baseline": "Accuracy_Baseline_mean", "Reweighing": "Accuracy_Reweighing_mean",
           "GridSearch": "Accuracy_GridSearch_mean", "ThresholdOpt": "Accuracy_ThresholdOpt_mean"}

for method, mdf in merged.items():
    mdf_sorted = mdf.sort_values("g_female")
    ax.plot(mdf_sorted["DI_Female_mean"], mdf_sorted[acc_col[method]],
            marker="o", markersize=4, color=METHOD_COLORS[method], label=METHOD_LABELS[method], linewidth=1.6)
    # annotate the fair (g=0) endpoint
    first = mdf_sorted.iloc[0]
    ax.annotate("g=0", (first["DI_Female_mean"], first[acc_col[method]]),
                textcoords="offset points", xytext=(4, 4), fontsize=7, color=METHOD_COLORS[method])

ax.axvline(1.0, color="black", linestyle="--", linewidth=1, alpha=0.4)
ax.set_xlabel("Disparate Impact Ratio (Female) — 1.0 = perfect parity")
ax.set_ylabel("Accuracy")
ax.set_title("Fairness-Accuracy Trade-off (Female), across the Discrimination Sweep")
ax.legend(loc="best", fontsize=8)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig4_fairness_accuracy_tradeoff.png", bbox_inches="tight")
plt.show()


In [ ]:
# Figure 5: GridSearch stability — DI_Female SD, grid_size=10 vs grid_size=50
fig, ax = plt.subplots(figsize=(8, 4.5))
width = 0.02
ax.bar(gridsearch_coarse_summary.g_female - width/2, gridsearch_coarse_summary.DI_Female_sd,
       width=width, label="grid_size=10 (fairlearn default)", color="#d62728", alpha=0.8)
ax.bar(gridsearch_summary.g_female + width/2, gridsearch_summary.DI_Female_sd,
       width=width, label="grid_size=50 (chosen)", color=METHOD_COLORS["GridSearch"], alpha=0.8)
ax.set_xlabel("g_female (sweep parameter)")
ax.set_ylabel("DI_Female SD (25 repeats)")
ax.set_title("GridSearch Stability: grid_size=10 vs. grid_size=50")
ax.legend(loc="best", fontsize=8)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/fig5_gridsearch_stability_comparison.png", bbox_inches="tight")
plt.show()

print(f"\nAll figures saved under {FIG_DIR}/")
print("Files:", sorted(os.listdir(FIG_DIR)))